# Calculate relative RPKM for closed regions

Calculate relative repair for H3K9me3 and H3K27me3 regions using the 1-minute RPKM as the reference.

In [2]:
from pathlib import Path
import gc
import pandas as pd

BASE_DIR = Path('/cta/users/guneyn23')
RPKM_DIR = BASE_DIR / 'peak_center_20kb/rpkm'
OUTPUT_DIR = BASE_DIR / 'relative_repair/closed_regions'

COLUMNS = ['chrom', 'start', 'end', 'peak', 'count', 'rpkm']
REGIONS = ['H3K9me3', 'H3K27me3']

SAMPLES = {
    'CPD': {
        '15m': 'R3Hela_15mCPD_TAGCTT_S2_hg38_primary_assembly_DS',
        '30m': 'R3Hela_30mCPD_GGCTAC_S8_hg38_primary_assembly_DS',
        '1h': 'R3Hela_1hCPD_CTTGTA_S4_hg38_primary_assembly_DS',
        '4h': 'R3Hela_4hCPD_AGTCAA_S10_hg38_primary_assembly_DS',
        '8h': 'R3Hela_8hCPD_AGTTCC_S12_hg38_primary_assembly_DS',
    },
    '64': {
        '15m': 'R3Hela_15m64_TTAGGC_S1_hg38_primary_assembly_DS',
        '30m': 'R3Hela_30m64_TGACCA_S7_hg38_primary_assembly_DS',
        '1h': 'R3Hela_1h64_ACAGTG_S3_hg38_primary_assembly_DS',
        '4h': 'R3Hela_4h64_GCCAAT_S9_hg38_primary_assembly_DS',
        '8h': 'R3Hela_8h64_CAGATC_S11_hg38_primary_assembly_DS',
    },
}

REFERENCE_FILES = {
    'H3K9me3': {
        'CPD': {
            'real': BASE_DIR / 'rpkm/CPD_rpkm/close_rpkm/H3k9me3_real_rpkm.bed',
            'sim': BASE_DIR / 'rpkm/CPD_rpkm/close_rpkm/H3k9me3_sim_rpkm.bed',
        },
        '64': {
            'real': BASE_DIR / 'rpkm/64_rpkm/H3K9me3_real_64_rpkm.bed',
            'sim': BASE_DIR / 'rpkm/64_rpkm/H3K9me3_simulated_64_rpkm.bed',
        },
    },
    'H3K27me3': {
        'CPD': {
            'real': BASE_DIR / 'rpkm/CPD_rpkm/close_rpkm/H3k27me3_real_rpkm.bed',
            'sim': BASE_DIR / 'rpkm/CPD_rpkm/close_rpkm/H3k27me3_sim_rpkm.bed',
        },
        '64': {
            'real': BASE_DIR / 'rpkm/64_rpkm/H3K27me3_real_64_rpkm.bed',
            'sim': BASE_DIR / 'rpkm/64_rpkm/H3K27me3_simulated_64_rpkm.bed',
        },
    },
}


In [3]:
def make_relative_file(reference_file, timepoint_file, output_file):
    reference = pd.read_csv(reference_file, sep='\t', header=None, names=COLUMNS)
    timepoint = pd.read_csv(timepoint_file, sep='\t', header=None, names=COLUMNS)

    result = timepoint.iloc[:, :5].copy()
    result['relative_rpkm'] = (reference['rpkm'] - timepoint['rpkm']) / reference['rpkm']
    result.to_csv(
        output_file,
        sep='\t',
        index=False,
        header=False,
        float_format='%.2f',
    )
    print(f'Written: {output_file}', flush=True)
    del reference, timepoint, result
    gc.collect()

## Run all calculations

This cell writes 40 relative-RPKM files and overwrites files with matching names.

In [3]:
for region in REGIONS:
    region_output_dir = OUTPUT_DIR / region
    region_output_dir.mkdir(parents=True, exist_ok=True)

    for damage_type, timepoints in SAMPLES.items():
        for data_type in ['real', 'sim']:
            reference_file = REFERENCE_FILES[region][damage_type][data_type]
            suffix = '' if data_type == 'real' else '_sim'

            for timepoint, sample in timepoints.items():
                timepoint_file = (
                    RPKM_DIR / region / f'{sample}{suffix}_{region}_400windows_rpkm.bed'
                )
                output_file = (
                    region_output_dir
                    / f'{damage_type}_{timepoint}_{data_type}_relative_rpkm.tsv'
                )
                make_relative_file(reference_file, timepoint_file, output_file)

Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_15m_real_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_30m_real_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_1h_real_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_4h_real_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_8h_real_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_15m_sim_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_30m_sim_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_1h_sim_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_4h_sim_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K9me3/CPD_8h_sim_relative_rpkm.tsv
Written: /cta/users/g

ZeroDivisionError: float division by zero

In [7]:

region = 'H3K27me3'
damage_type = '64'
data_type = 'sim'
rpkm_dir = BASE_DIR / 'peak_center_20kb/rpkm/H3K27me3'
remaining_timepoints = ['30m', '1h', '4h', '8h']

region_output_dir = OUTPUT_DIR / region
region_output_dir.mkdir(parents=True, exist_ok=True)
reference_file = REFERENCE_FILES[region][damage_type][data_type]

for timepoint in remaining_timepoints:
    sample = SAMPLES[damage_type][timepoint]
    timepoint_file = (
        rpkm_dir / f'{sample}_sim_{region}_400windows_rpkm.bed'
    )

    output_file = (
        region_output_dir / f'{damage_type}_{timepoint}_{data_type}_relative_rpkm.tsv'
    )
    make_relative_file(reference_file, timepoint_file, output_file)

Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K27me3/64_30m_sim_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K27me3/64_1h_sim_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K27me3/64_4h_sim_relative_rpkm.tsv
Written: /cta/users/guneyn23/relative_repair/closed_regions/H3K27me3/64_8h_sim_relative_rpkm.tsv
